# NEDI x2 Colab Pilot

This notebook runs one Set5 image through native NEDI x2. It is a correctness and runtime pilot, not the final NEDI evaluation.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f'Repository ready: {REPO_ROOT}')

In [ ]:
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.evaluation.bicubic import BicubicEvaluationConfig, evaluate_bicubic_image
from app.evaluation.images import load_rgb_image, pair_image_paths
from app.evaluation.nedi import NEDIEvaluationConfig, evaluate_nedi_image
from app.evaluation.experiment import write_results_csv
print('NEDI evaluator imported successfully.')

In [ ]:
from datetime import UTC, datetime

from app.evaluation.data_validation import validate_prepared_dataset

DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
RUN_ID = datetime.now(UTC).strftime('%Y%m%d_%H%M%S_utc')
RUN_ROOT = DATA_ROOT / 'results' / 'final_nedi' / 'pilot' / RUN_ID
METRICS_ROOT = RUN_ROOT / 'metrics'
BICUBIC_IMAGES_ROOT = RUN_ROOT / 'images' / 'bicubic'
NEDI_IMAGES_ROOT = RUN_ROOT / 'images' / 'nedi'

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset root not found: {DATA_ROOT}')

validation = validate_prepared_dataset('Set5', 2, DATA_ROOT)
print(f'VALID: {validation.dataset} x{validation.scale} has {validation.image_count} pairs.')
print(f'Pilot output: {RUN_ROOT}')

In [ ]:
# Both methods receive the exact same first Set5 HR/LR image pair.
# These shorter timing settings are only for this pilot.
hr_path, lr_path = pair_image_paths(
    validation.hr_directory,
    validation.lr_directory,
)[0]

bicubic_config = BicubicEvaluationConfig(
    dataset='Set5',
    scale=2,
    warmup_runs=1,
    timed_runs=3,
)
nedi_config = NEDIEvaluationConfig(
    dataset='Set5',
    scale=2,
    window_size=8,
    edge_threshold=8.0,
    warmup_runs=1,
    timed_runs=3,
)

bicubic_record = evaluate_bicubic_image(
    hr_path,
    lr_path,
    bicubic_config,
    sr_output_dir=BICUBIC_IMAGES_ROOT,
)
nedi_record = evaluate_nedi_image(
    hr_path,
    lr_path,
    nedi_config,
    sr_output_dir=NEDI_IMAGES_ROOT,
)
bicubic_csv = write_results_csv(
    [bicubic_record],
    METRICS_ROOT / 'Set5_x2_bicubic_pilot.csv',
)
nedi_csv = write_results_csv(
    [nedi_record],
    METRICS_ROOT / 'Set5_x2_nedi_pilot.csv',
)
print(f'Bicubic pilot result: {bicubic_csv}')
print(f'NEDI pilot result: {nedi_csv}')

In [ ]:
print(f"Image: {nedi_record['image']}")
for metric in ('psnr_y', 'ssim_y', 'psnr_rgb', 'ssim_rgb'):
    bicubic_value = bicubic_record[metric]
    nedi_value = nedi_record[metric]
    print(
        f'{metric}: bicubic={bicubic_value:.6f} | '
        f'nedi={nedi_value:.6f} | '
        f'difference={nedi_value - bicubic_value:+.6f}'
    )

for method, record in (('bicubic', bicubic_record), ('nedi', nedi_record)):
    print(f"{method} latency_mean_ms: {record['latency_mean_ms']:.2f}")

In [ ]:
import matplotlib.pyplot as plt

hr_image = load_rgb_image(hr_path)
bicubic_image = load_rgb_image(
    BICUBIC_IMAGES_ROOT / f'{hr_path.stem}_bicubic_x2.png'
)
nedi_image = load_rgb_image(NEDI_IMAGES_ROOT / f'{hr_path.stem}_nedi_x2.png')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for axis, title, image in (
    (axes[0], 'Original HR', hr_image),
    (axes[1], 'Bicubic', bicubic_image),
    (axes[2], 'NEDI', nedi_image),
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis('off')

plt.tight_layout()
plt.show()
print(f'Output folder: {RUN_ROOT}')